# SSpace

This is wassname's experimental S-space method. Rank keeps the largest absolute S-space mean contrasts. The default absolute cosine gate depends on each token. [Project](https://apartresearch.com/project/sspace-steering-for-evalawareness-control-in-reasoning-models-7j1i).

We use a random CPU Llama and token ids to demonstrate calibration and execution without downloads. The output checks establish a numerical effect, not behavior control. Run with the repository environment (`uv sync --extra all`).

Implementation source: [steering-lite 0a064ba](https://github.com/wassname/steering-lite/tree/0a064ba0c23a4998637ff41c5ab0fb5ca50a4271/src/steering_lite/variants).

Authored by PI/OpenAI.

In [1]:
from steerability.algorithms.state_control.sspace.control import SSpace
import torch
from tokenizers import Tokenizer, models, pre_tokenizers
from transformers import LlamaConfig, LlamaForCausalLM, PreTrainedTokenizerFast
from steerability.algorithms.core.steering_pipeline import SteeringPipeline

torch.manual_seed(12)
torch.set_num_threads(1)
model = LlamaForCausalLM(LlamaConfig(
    vocab_size=16, hidden_size=16, intermediate_size=32,
    num_hidden_layers=2, num_attention_heads=2, num_key_value_heads=2,
)).eval()
raw = Tokenizer(models.WordLevel({"<pad>": 0, "<s>": 1, "</s>": 2, "yes": 3, "no": 4}, unk_token="<pad>"))
raw.pre_tokenizer = pre_tokenizers.Whitespace()
tokenizer = PreTrainedTokenizerFast(tokenizer_object=raw, pad_token="<pad>", bos_token="<s>", eos_token="</s>")
positive_ids = torch.tensor([[1, 3, 5], [1, 6, 7], [1, 8, 9], [1, 10, 11]])
negative_ids = torch.tensor([[1, 4, 6], [1, 7, 8], [1, 9, 10], [1, 11, 12]])


## Calibration

Each row is one prompt. These equal-length examples have no padding, so position `-1` is the last prompt token. For padded data, gather the last position whose attention mask is one. Capture from the same frozen model that will be steered. CorDA requires paired rows; Linear-AcT requires equal counts. The fit runs during `steer()`, and the inference edit applies at every token, including the prompt.

In [2]:
target = "model.layers.0.mlp.down_proj"
module = model.get_submodule(target)

def capture_last(ids):
    captured = []
    def capture(module, inputs, output):
        captured.append(output[:, -1, :].detach().cpu())
    handle = module.register_forward_hook(capture)
    try:
        with torch.no_grad():
            model(input_ids=ids, attention_mask=torch.ones_like(ids))
    finally:
        handle.remove()
    return captured[0]

positive, negative = capture_last(positive_ids), capture_last(negative_ids)

In [3]:
control = SSpace(positive_outputs={target: positive}, negative_outputs={target: negative}, rank=4, strength=0.5)

## Apply

Scoring the same continuation shows a nonzero change. Scoring again without the control checks that hooks do not remain on the model. For a pretrained model, use contrastive text and evaluate held-out behavior separately.

In [4]:
baseline = SteeringPipeline(model=model, tokenizer=tokenizer, controls=[])
pipeline = SteeringPipeline(model=model, tokenizer=tokenizer, controls=[control])
baseline.steer()
pipeline.steer()
query, reference = torch.tensor([[1, 3, 4]]), torch.tensor([[5, 6]])
base_scores = baseline.compute_logprobs(query, ref_output_ids=reference)
steered_scores = pipeline.compute_logprobs(query, ref_output_ids=reference)
assert not torch.allclose(base_scores, steered_scores)
torch.testing.assert_close(baseline.compute_logprobs(query, ref_output_ids=reference), base_scores)
print("Maximum log-probability change:", (steered_scores - base_scores).abs().max().item())
print("Generated token ids:", pipeline.generate(input_ids=query, max_new_tokens=3, do_sample=False).tolist())

Maximum log-probability change: 0.05019950866699219
Generated token ids: [[10, 8, 8]]
